In [1]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install('ultralytics>=8.2.0')
pip_install('wandb>=0.17.0')
pip_install('matplotlib', 'numpy', 'pillow', 'pyyaml', 'tabulate')
print('✓ Semua dependensi berhasil diinstall')

import wandb, os
WANDB_API_KEY = "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"  
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
wandb.login(key=WANDB_API_KEY, relogin=True)
print('✓ WandB login berhasil')

✓ Semua dependensi berhasil diinstall


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/rof/.netrc
wandb: Currently logged in as: delimabrpurba72 (delimabrpurba72-universitas-prasetiya-mulya) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ WandB login berhasil


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/rof/.netrc
wandb: Currently logged in as: delimabrpurba72 (delimabrpurba72-universitas-prasetiya-mulya) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ WandB login berhasil


## Cell 2 — Konfigurasi Path, Dataset, GPU & Parameter Dasar

In [2]:
from pathlib import Path
import yaml, os, sys, torch, json, math
import numpy as np

# ── Path dataset (sama persis dengan Train Dataset Terbaru.ipynb) ─────────────
Data_Train      = Path(r"/home/rof/Delima/Delima/CLEAN DATASET /Balanced_Dataset/train")
Data_Test       = Path(r"/home/rof/Delima/Delima/CLEAN DATASET /Balanced_Dataset/test")
Data_Validation = Path(r"/home/rof/Delima/Delima/CLEAN DATASET /Balanced_Dataset/validation")

CLASS_NAMES = [
    'atrophic_scar', 'comedo', 'hypertrophic_scar', 'melasma',
    'nevus', 'nodule', 'other', 'papule', 'pustule'
]
NUM_CLASSES = len(CLASS_NAMES)
NOTEBOOK_ROOT = Path(r"/home/rof/Delima/Delima/Eksperimen 5")
OUTPUT_DIR = NOTEBOOK_ROOT / "runs_v5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Generate data.yaml secara dinamis dari path eksplisit ─────────────────────
cfg = {
    'train': str(Data_Train / 'images'),
    'val':   str(Data_Validation / 'images'),
    'test':  str(Data_Test / 'images'),
    'nc':    NUM_CLASSES,
    'names': CLASS_NAMES,
}

DATA_YAML = OUTPUT_DIR / 'data_runtime.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

DEVICE = 0 if torch.cuda.is_available() else 'cpu'

print(f'Train    : {cfg["train"]}')
print(f'Val      : {cfg["val"]}')
print(f'Test     : {cfg["test"]}')
print(f'Kelas    : {NUM_CLASSES} → {CLASS_NAMES}')
print(f'data.yaml: {DATA_YAML}')
print(f'Output   : {OUTPUT_DIR}')
print(f'GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

WANDB_PROJECT = "acne-detection-progressive"
WANDB_ENTITY  = None

print('\n✓ Konfigurasi siap')

Train    : /home/rof/Delima/Delima/CLEAN DATASET /Balanced_Dataset/train/images
Val      : /home/rof/Delima/Delima/CLEAN DATASET /Balanced_Dataset/validation/images
Test     : /home/rof/Delima/Delima/CLEAN DATASET /Balanced_Dataset/test/images
Kelas    : 9 → ['atrophic_scar', 'comedo', 'hypertrophic_scar', 'melasma', 'nevus', 'nodule', 'other', 'papule', 'pustule']
data.yaml: /home/rof/Delima/Delima/Eksperimen 5/runs_v5/data_runtime.yaml
Output   : /home/rof/Delima/Delima/Eksperimen 5/runs_v5
GPU      : NVIDIA GeForce RTX 3060

✓ Konfigurasi siap


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ═══════════════════════════════════════════════════════════════════════════════
# 1.  EFFICIENT MULTI-SCALE ATTENTION (EMA)
# ═══════════════════════════════════════════════════════════════════════════════

class EMA(nn.Module):
    """Efficient Multi-Scale Attention — plug-in attention module untuk neck YOLO."""

    def __init__(self, channels: int, factor: int = 8):
        super().__init__()
        self.groups = factor
        assert channels // self.groups > 0, "channels harus kelipatan factor"
        self.softmax  = nn.Softmax(dim=-1)
        self.agp      = nn.AdaptiveAvgPool2d((1, 1))
        self.pool_h   = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w   = nn.AdaptiveAvgPool2d((1, None))
        self.gn       = nn.GroupNorm(channels // self.groups, channels // self.groups)
        self.conv1x1  = nn.Conv2d(channels // self.groups, channels // self.groups,
                                   kernel_size=1, bias=False)
        self.conv3x3  = nn.Conv2d(channels // self.groups, channels // self.groups,
                                   kernel_size=3, padding=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.size()
        group_x = x.reshape(b * self.groups, -1, h, w)
        x_h = self.pool_h(group_x)
        x_w = self.pool_w(group_x).permute(0, 1, 3, 2)
        hw  = self.conv1x1(torch.cat([x_h, x_w], dim=2))
        x_h, x_w = torch.split(hw, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        a1  = torch.sigmoid(self.gn(x_h))
        a2  = torch.sigmoid(self.gn(x_w))
        group_x = group_x * a1.expand_as(group_x) * a2.expand_as(group_x)
        x1  = self.agp(group_x)
        x2  = self.conv3x3(group_x).reshape(b * self.groups, -1, 1)
        x11 = self.softmax(x1.reshape(b * self.groups, -1, 1))
        x12 = x1.reshape(b * self.groups, -1, 1)
        x21 = self.softmax(x2)
        x22 = x2
        weights = (torch.matmul(x11.transpose(1,2), x12) +
                   torch.matmul(x21.transpose(1,2), x22)).reshape(b * self.groups, 1, h, w)
        out = group_x * weights.sigmoid().expand_as(group_x)
        return out.reshape(b, c, h, w)


# ═══════════════════════════════════════════════════════════════════════════════
# 2.  POWERFUL IoU (PIoU) LOSS
# ═══════════════════════════════════════════════════════════════════════════════

def piou_loss(pred: torch.Tensor, target: torch.Tensor,
              alpha: float = 1.0, beta: float = 1.0) -> torch.Tensor:
    """
    PIoU loss untuk axis-aligned bounding boxes.
    pred, target: (N, 4) format (x1, y1, x2, y2)
    """
    inter_x1 = torch.max(pred[:, 0], target[:, 0])
    inter_y1 = torch.max(pred[:, 1], target[:, 1])
    inter_x2 = torch.min(pred[:, 2], target[:, 2])
    inter_y2 = torch.min(pred[:, 3], target[:, 3])
    inter_w  = (inter_x2 - inter_x1).clamp(min=0)
    inter_h  = (inter_y2 - inter_y1).clamp(min=0)
    inter    = inter_w * inter_h
    area_p   = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])
    area_t   = (target[:, 2] - target[:, 0]) * (target[:, 3] - target[:, 1])
    union    = area_p + area_t - inter + 1e-7
    iou      = inter / union

    enc_x1   = torch.min(pred[:, 0], target[:, 0])
    enc_y1   = torch.min(pred[:, 1], target[:, 1])
    enc_x2   = torch.max(pred[:, 2], target[:, 2])
    enc_y2   = torch.max(pred[:, 3], target[:, 3])
    c2       = (enc_x2 - enc_x1) ** 2 + (enc_y2 - enc_y1) ** 2 + 1e-7

    p_cx = (pred[:, 0] + pred[:, 2]) / 2
    p_cy = (pred[:, 1] + pred[:, 3]) / 2
    t_cx = (target[:, 0] + target[:, 2]) / 2
    t_cy = (target[:, 1] + target[:, 3]) / 2
    rho2 = (p_cx - t_cx) ** 2 + (p_cy - t_cy) ** 2

    corner_dist = (
        (pred[:, 0] - target[:, 0]) ** 2 + (pred[:, 1] - target[:, 1]) ** 2 +
        (pred[:, 2] - target[:, 2]) ** 2 + (pred[:, 1] - target[:, 1]) ** 2 +
        (pred[:, 0] - target[:, 0]) ** 2 + (pred[:, 3] - target[:, 3]) ** 2 +
        (pred[:, 2] - target[:, 2]) ** 2 + (pred[:, 3] - target[:, 3]) ** 2
    )

    piou = iou - rho2 / c2 - alpha * corner_dist / (c2 + 1e-7) * beta
    return (1 - piou).unsqueeze(-1)


# ═══════════════════════════════════════════════════════════════════════════════
# 3.  PATCH ultralytics — PIoU loss + EMA injection
# ═══════════════════════════════════════════════════════════════════════════════

from ultralytics import YOLO
from ultralytics.nn.modules import C2f
from ultralytics.nn.modules.block import SPPF
import ultralytics.utils.loss as ult_loss
from ultralytics.utils.metrics import bbox_iou
from ultralytics.utils.tal import bbox2dist

if not hasattr(ult_loss.BboxLoss, '_truly_original_forward'):
    ult_loss.BboxLoss._truly_original_forward = ult_loss.BboxLoss.forward

def _piou_bbox_loss_forward(self, pred_dist, pred_bboxes, anchor_points,
                             target_bboxes, target_scores, target_scores_sum, fg_mask,
                             imgsz=None, stride=None):
    """Override BboxLoss.forward dengan PIoU. Kompatibel Ultralytics 8.4.x."""
    weight = target_scores.sum(-1)[fg_mask].unsqueeze(-1)
    iou = bbox_iou(pred_bboxes[fg_mask], target_bboxes[fg_mask], xywh=False, CIoU=True)

    piou_val = piou_loss(pred_bboxes[fg_mask], target_bboxes[fg_mask])
    loss_iou = ((1.0 - iou) * 0.3 + piou_val * 0.7) * weight
    loss_iou = loss_iou.sum() / target_scores_sum

    if self.dfl_loss is not None:
        target_ltrb = bbox2dist(anchor_points, target_bboxes, self.dfl_loss.reg_max - 1)
        loss_dfl = self.dfl_loss(
            pred_dist[fg_mask].view(-1, self.dfl_loss.reg_max),
            target_ltrb[fg_mask]
        ) * weight
        loss_dfl = loss_dfl.sum() / target_scores_sum
    else:
        if imgsz is not None and stride is not None:
            tl = bbox2dist(anchor_points, target_bboxes) * stride
            tl[..., 0::2] /= imgsz[1]
            tl[..., 1::2] /= imgsz[0]
            pd = pred_dist * stride
            pd[..., 0::2] /= imgsz[1]
            pd[..., 1::2] /= imgsz[0]
            loss_dfl = (F.l1_loss(pd[fg_mask], tl[fg_mask], reduction='none')
                        .mean(-1, keepdim=True) * weight)
            loss_dfl = loss_dfl.sum() / target_scores_sum
        else:
            loss_dfl = torch.tensor(0., device=pred_dist.device)
    return loss_iou, loss_dfl

ult_loss.BboxLoss.forward = _piou_bbox_loss_forward
print('✓ PIoU berhasil di-patch ke BboxLoss (Ultralytics 8.4.x kompatibel)')


class C2fWithEMA(nn.Module):
    """C2f + EMA attention wrapper untuk neck YOLOv8."""

    def __init__(self, c2f_module: C2f):
        super().__init__()
        self.c2f = c2f_module
        ch = c2f_module.cv2.conv.out_channels
        factor = 8
        while ch % factor != 0 and factor > 1:
            factor //= 2
        self.ema = EMA(channels=ch, factor=factor)

    def forward(self, x):
        return self.ema(self.c2f(x))


def inject_ema_to_neck(model):
    """
    Deteksi akhir backbone secara otomatis via SPPF layer,
    lalu ganti semua C2f di neck dengan C2fWithEMA.
    Kompatibel dengan YOLOv8 n/s/m/l/x.
    """
    backbone_end = None
    for i, layer in enumerate(model.model.model):
        if isinstance(layer, SPPF):
            backbone_end = i
            break
    neck_start = (backbone_end or 9) + 1

    injected = 0
    for i, layer in enumerate(model.model.model):
        if i >= neck_start and isinstance(layer, C2f):
            model.model.model[i] = C2fWithEMA(layer)
            injected += 1
    print(f'✓ EMA disematkan ke {injected} C2f layer di neck (backbone_end={backbone_end}, neck_start={neck_start})')
    return model


# ═══════════════════════════════════════════════════════════════════════════════
# 4.  FOCAL LOSS GAMMA PATCH
#     Ultralytics 8.4.x tidak menerima fl_gamma sebagai argumen model.train().
#     Kita implementasikan via monkey-patch pada v8DetectionLoss.__init__
#     agar mengganti self.bce dengan versi focal-loss-aware.
# ═══════════════════════════════════════════════════════════════════════════════

_FL_GAMMA = 0.0

def set_fl_gamma(gamma: float):
    """Set global focal loss gamma sebelum memanggil model.train()."""
    global _FL_GAMMA
    _FL_GAMMA = gamma


class FocalBCEWithLogitsLoss(nn.Module):
    """BCEWithLogitsLoss dengan focal loss modulation."""
    def __init__(self, gamma=0.0, reduction='none'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, pred, target):
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        if self.gamma > 0:
            p_t = torch.exp(-bce)
            bce = bce * ((1 - p_t) ** self.gamma)
        if self.reduction == 'mean':
            return bce.mean()
        elif self.reduction == 'sum':
            return bce.sum()
        return bce


if not hasattr(ult_loss.v8DetectionLoss, '_truly_original_init'):
    ult_loss.v8DetectionLoss._truly_original_init = ult_loss.v8DetectionLoss.__init__

def _patched_v8det_init(self, *args, **kwargs):
    ult_loss.v8DetectionLoss._truly_original_init(self, *args, **kwargs)
    if _FL_GAMMA > 0:
        self.bce = FocalBCEWithLogitsLoss(gamma=_FL_GAMMA, reduction='none')

ult_loss.v8DetectionLoss.__init__ = _patched_v8det_init
print('✓ Focal Loss Gamma patch siap (set via set_fl_gamma() sebelum training)')


print('\n✓ Modul EMA, PIoU & Focal Loss siap.')

✓ PIoU berhasil di-patch ke BboxLoss (Ultralytics 8.4.x kompatibel)
✓ Focal Loss Gamma patch siap (set via set_fl_gamma() sebelum training)

✓ Modul EMA, PIoU & Focal Loss siap.


In [4]:
import json

# ═══════════════════════════════════════════════════════════════════════════════
# HYPERPARAMETERS — langsung dari konfigurasi (tanpa tuning)
# ═══════════════════════════════════════════════════════════════════════════════

best_params = {
    "optimizer":     "AdamW",
    "lr0":           0.002064871781726893,
    "lrf":           0.029206531194913628,
    "weight_decay":  0.001245635683275296,
    "momentum":      0.9451593759443165,
    "batch":         16,
    "box":           5.852405705833931,
    "cls":           0.9689743502057859,
    "dfl":           1.0880571498971556,
    "warmup_epochs": 3.3403211882493458,
    "cos_lr":        False,
    "fl_gamma":      0.0,
}

BEST_PARAMS_JSON = OUTPUT_DIR / 'best_params.json'
with open(BEST_PARAMS_JSON, 'w') as f:
    json.dump(best_params, f, indent=2)

print(f'{"="*50}')
print(f' Hyperparameters (fixed — tanpa tuning)')
print(f'{"="*50}')
for k, v in best_params.items():
    print(f'  {k:20s}: {v}')
print(f'\n✓ best_params tersimpan di {BEST_PARAMS_JSON}')

 Hyperparameters (fixed — tanpa tuning)
  optimizer           : AdamW
  lr0                 : 0.002064871781726893
  lrf                 : 0.029206531194913628
  weight_decay        : 0.001245635683275296
  momentum            : 0.9451593759443165
  batch               : 16
  box                 : 5.852405705833931
  cls                 : 0.9689743502057859
  dfl                 : 1.0880571498971556
  warmup_epochs       : 3.3403211882493458
  cos_lr              : False
  fl_gamma            : 0.0

✓ best_params tersimpan di /home/rof/Delima/Delima/Eksperimen 5/runs_v5/best_params.json


[W 2026-04-14 19:53:24,946] Trial 1 failed with parameters: {'optimizer': 'SGD', 'lr0': 0.0004678719265016201, 'lrf': 0.06252287916406217, 'momentum': 0.866739263278245, 'weight_decay': 0.0003135775732257748, 'batch': 16, 'box': 8.925879806965067, 'cls': 0.6394454296692116, 'dfl': 1.6741985453031396, 'fl_gamma': 1.684829137724085, 'warmup_epochs': 2.139351238159993, 'cos_lr': True, 'close_mosaic': 11, 'mixup': 0.28466566117599995} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/rof/.local/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_1207909/469428303.py", line 92, in objective
    results = model.train(
  File "/home/rof/.local/lib/python3.10/site-packages/ultralytics/engine/model.py", line 777, in train
    self.trainer.train()
  File "/home/rof/.local/lib/python3.10/site-packages/ultralytics/engine/trainer.py", line 244, in train
    self.

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x747092e69390>> (for post_run_cell), with arguments args (<ExecutionResult object at 747220331b10, execution_count=6 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 747220331bd0, raw_cell="import optuna
import wandb
import json
from ultral.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a2273736820726f664031302e33302e3230302e3430227d/home/rof/Delima/Delima/CLEAN%20DATASET%20/Train5_Progressive_Tuning_3.ipynb#X24sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

## Cell 4b — Verifikasi best_params

Cell ini memverifikasi bahwa `best_params` sudah terdefinisi dengan benar dari Cell 4.

In [5]:
assert 'best_params' in dir() and best_params, \
    'best_params belum terdefinisi. Jalankan Cell 4 terlebih dahulu.'

required_keys = ['optimizer', 'lr0', 'lrf', 'weight_decay', 'momentum',
                 'batch', 'box', 'cls', 'dfl', 'warmup_epochs', 'cos_lr', 'fl_gamma']

missing = [k for k in required_keys if k not in best_params]
assert not missing, f'Key berikut hilang dari best_params: {missing}'

print('✓ best_params terverifikasi — semua key lengkap')
print(f'  Optimizer : {best_params["optimizer"]}')
print(f'  LR0       : {best_params["lr0"]}')
print(f'  Batch     : {best_params["batch"]}')
print(f'  Box/Cls/DFL: {best_params["box"]:.4f} / {best_params["cls"]:.4f} / {best_params["dfl"]:.4f}')

✓ best_params terverifikasi — semua key lengkap
  Optimizer : AdamW
  LR0       : 0.002064871781726893
  Batch     : 16
  Box/Cls/DFL: 5.8524 / 0.9690 / 1.0881


Process Process-1:
Process Process-7:
Process Process-5:
Process Process-6:
Process Process-8:
Process Process-3:
Process Process-2:
Traceback (most recent call last):
Traceback (most recent call last):
Process Process-4:
Traceback (most recent call last):
  File "/home/rof/.conda/envs/yolo_gpu/lib/python3.10/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
Traceback (most recent call last):
  File "/home/rof/.conda/envs/yolo_gpu/lib/python3.10/multiprocessing/util.py", line 360, in _exit_function
    _run_finalizers()
Traceback (most recent call last):
  File "/home/rof/.conda/envs/yolo_gpu/lib/python3.10/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/home/rof/.conda/envs/yolo_gpu/lib/python3.10/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/home/rof/.conda/envs/yolo_gpu/lib/python3.10/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/rof/.

## Cell 5 — Progressive Training: YOLOv8n → YOLOv8s

Menggunakan `best_params` (fixed hyperparameters) untuk melatih kedua varian model.
Setiap varian mendapat EMA injection dan PIoU loss yang sama.

In [6]:
import json, gc, torch
import wandb
from ultralytics import YOLO

# ─── Verifikasi best_params ──────────────────────────────────────────────────
assert 'best_params' in dir() and best_params, \
    'best_params belum terdefinisi. Jalankan Cell 4 terlebih dahulu.'

print('Best params:')
for k, v in best_params.items():
    print(f'  {k:20s}: {v}')

# ─── Konfigurasi Progressive Training ────────────────────────────────────────
MODEL_VARIANTS = [
    ('yolov8n', 'yolov8n.pt', 16),
    ('yolov8s', 'yolov8s.pt', 16),
]

PROGRESSIVE_EPOCHS = 150
PROGRESSIVE_IMGSZ  = 640

EXCLUDE_FROM_TRAIN = {'batch', 'fl_gamma'}
hp_for_train = {k: v for k, v in best_params.items() if k not in EXCLUDE_FROM_TRAIN}

LR_SCALE = {
    'yolov8n': 1.0,
    'yolov8s': 0.7,
}

progressive_results = {}

for variant_name, weights, batch_size in MODEL_VARIANTS:
    print(f'\n{"="*60}')
    print(f' Progressive Training: {variant_name.upper()} (batch={batch_size})')
    print(f'{"="*60}')

    fl_gamma_val = best_params.get('fl_gamma', 0.0)
    set_fl_gamma(fl_gamma_val)

    scale = LR_SCALE[variant_name]
    hp_scaled = hp_for_train.copy()
    hp_scaled['lr0'] = hp_for_train['lr0'] * scale

    run = wandb.init(
        project = WANDB_PROJECT,
        entity  = WANDB_ENTITY,
        name    = f'progressive_{variant_name}',
        group   = 'progressive_training_v3',
        config  = {
            'variant': variant_name,
            'batch': batch_size,
            'epochs': PROGRESSIVE_EPOCHS,
            **hp_scaled,
            'fl_gamma': fl_gamma_val,
            'lr_scale': scale,
        },
        reinit = True,
    )

    model = YOLO(weights)
    inject_ema_to_neck(model)

    results = model.train(
        data          = str(DATA_YAML),
        epochs        = PROGRESSIVE_EPOCHS,
        imgsz         = PROGRESSIVE_IMGSZ,
        batch         = batch_size,
        device        = DEVICE,
        project       = str(OUTPUT_DIR),
        name          = f'progressive_{variant_name}',
        exist_ok      = True,
        verbose       = True,
        workers       = 4,
        patience      = 25,
        degrees       = 10.0,
        flipud        = 0.3,
        dropout       = 0.1,
        amp           = True,
        save          = True,
        plots         = True,
        deterministic = True,
        seed          = 42,
        **hp_scaled,
    )

    rd = results.results_dict
    progressive_results[variant_name] = {
        'map50':    float(rd.get('metrics/mAP50(B)', 0.0)),
        'map50_95': float(rd.get('metrics/mAP50-95(B)', 0.0)),
        'recall':   float(rd.get('metrics/recall(B)', 0.0)),
        'precision':float(rd.get('metrics/precision(B)', 0.0)),
        'best_pt':  str(OUTPUT_DIR / f'progressive_{variant_name}' / 'weights' / 'best.pt'),
    }

    wandb.log(progressive_results[variant_name])
    run.finish()

    del model, results
    gc.collect()
    torch.cuda.empty_cache()
    print(f'✓ {variant_name} selesai: mAP50-95={progressive_results[variant_name]["map50_95"]:.4f}')

with open(OUTPUT_DIR / 'progressive_results.json', 'w') as f:
    json.dump(progressive_results, f, indent=2)

print('\n═══ Progressive Training Selesai ═══')
for vn, r in progressive_results.items():
    print(f'  {vn:10s}: mAP50={r["map50"]:.4f}  mAP50-95={r["map50_95"]:.4f}  recall={r["recall"]:.4f}  prec={r["precision"]:.4f}')

Best params:
  optimizer           : AdamW
  lr0                 : 0.002064871781726893
  lrf                 : 0.029206531194913628
  weight_decay        : 0.001245635683275296
  momentum            : 0.9451593759443165
  batch               : 16
  box                 : 5.852405705833931
  cls                 : 0.9689743502057859
  dfl                 : 1.0880571498971556
  warmup_epochs       : 3.3403211882493458
  cos_lr              : False
  fl_gamma            : 0.0

 Progressive Training: YOLOV8N (batch=16)


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


✓ EMA disematkan ke 4 C2f layer di neck (backbone_end=9, neck_start=10)
New https://pypi.org/project/ultralytics/8.4.37 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.22 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11908MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=5.852405705833931, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.9689743502057859, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/rof/Delima/Delima/Eksperimen 5/runs_v5/data_runtime.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.0880571498971556, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_w

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


map50,▁
map50_95,▁
precision,▁
recall,▁
best_pt,/home/rof/Delima/Del...
map50,0.76983
map50_95,0.54384
precision,0.75741
recall,0.73138


✓ yolov8n selesai: mAP50-95=0.5438

 Progressive Training: YOLOV8S (batch=16)


✓ EMA disematkan ke 4 C2f layer di neck (backbone_end=9, neck_start=10)
New https://pypi.org/project/ultralytics/8.4.37 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.22 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11908MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=5.852405705833931, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.9689743502057859, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/rof/Delima/Delima/Eksperimen 5/runs_v5/data_runtime.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.0880571498971556, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_w

KeyboardInterrupt: 

✓ EMA disematkan ke 4 C2f layer di neck (backbone_end=9, neck_start=10)
New https://pypi.org/project/ultralytics/8.4.36 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.22 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11908MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=12.82007021137776, cache=False, cfg=None, classes=None, close_mosaic=10, cls=4.277628208996668, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/rof/Delima/Delima/Eksperimen 5/runs_v5/data_runtime.yaml, degrees=0.0, deterministic=True, device=0, dfl=0.45850060607767884, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=350, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_widt

KeyboardInterrupt: 

## Cell 6 — Comparison Table: Pilih Varian Terbaik

Memuat kedua `best.pt` (YOLOv8n & YOLOv8s), jalankan validasi pada test set, dan tampilkan tabel perbandingan
per-class recall/precision + overall metrics. Varian terbaik dipilih secara programatis.

In [7]:
import json
import numpy as np
from ultralytics import YOLO
from tabulate import tabulate
import matplotlib.pyplot as plt

# ─── Load progressive results ────────────────────────────────────────────────
prog_json = OUTPUT_DIR / 'progressive_results.json'
if prog_json.exists():
    with open(prog_json) as f:
        progressive_results = json.load(f)
else:
    print('progressive_results.json tidak ditemukan. Jalankan Cell 5 dulu.')

# ─── Evaluasi ulang pada TEST set dengan per-class metrics ────────────────────
comparison = {}

for variant_name, info in progressive_results.items():
    best_pt = info['best_pt']
    print(f'\nEvaluasi {variant_name}: {best_pt}')

    model = YOLO(best_pt)
    metrics = model.val(
        data    = str(DATA_YAML),
        split   = 'test',
        batch   = 16,
        device  = DEVICE,
        verbose = False,
    )

    per_class_recall = metrics.box.r.tolist()      if hasattr(metrics.box.r, 'tolist')    else list(metrics.box.r)
    per_class_prec   = metrics.box.p.tolist()      if hasattr(metrics.box.p, 'tolist')    else list(metrics.box.p)
    per_class_ap50   = metrics.box.ap50.tolist()   if hasattr(metrics.box.ap50, 'tolist') else list(metrics.box.ap50)

    comparison[variant_name] = {
        'map50':       float(metrics.box.map50),
        'map50_95':    float(metrics.box.map),
        'mean_recall': float(np.mean(per_class_recall)),
        'min_recall':  float(np.min(per_class_recall)),
        'mean_prec':   float(np.mean(per_class_prec)),
        'per_class_recall': per_class_recall,
        'per_class_prec':   per_class_prec,
        'per_class_ap50':   per_class_ap50,
        'composite':   0.60 * float(metrics.box.map) + 0.40 * float(np.mean(per_class_recall)),
    }
    del model

# ─── Overall Comparison Table ─────────────────────────────────────────────────
print('\n' + '='*80)
print(' OVERALL COMPARISON (TEST SET)')
print('='*80)

headers = ['Variant', 'mAP50', 'mAP50-95', 'Mean Recall', 'Min Recall', 'Mean Prec', 'Composite']
rows = []
for vn, c in comparison.items():
    rows.append([
        vn,
        f'{c["map50"]:.4f}',
        f'{c["map50_95"]:.4f}',
        f'{c["mean_recall"]:.4f}',
        f'{c["min_recall"]:.4f}',
        f'{c["mean_prec"]:.4f}',
        f'{c["composite"]:.4f}',
    ])
print(tabulate(rows, headers=headers, tablefmt='grid'))

# ─── Per-Class Recall Table ───────────────────────────────────────────────────
print('\n' + '='*80)
print(' PER-CLASS RECALL (TEST SET)')
print('='*80)

pc_headers = ['Class'] + list(comparison.keys())
pc_rows = []
for i, cls_name in enumerate(CLASS_NAMES):
    row = [cls_name]
    for vn in comparison:
        r = comparison[vn]['per_class_recall']
        row.append(f'{r[i]:.4f}' if i < len(r) else 'N/A')
    pc_rows.append(row)
print(tabulate(pc_rows, headers=pc_headers, tablefmt='grid'))

# ─── Per-Class AP50 Table ─────────────────────────────────────────────────────
print('\n' + '='*80)
print(' PER-CLASS AP50 (TEST SET)')
print('='*80)

ap_rows = []
for i, cls_name in enumerate(CLASS_NAMES):
    row = [cls_name]
    for vn in comparison:
        ap = comparison[vn]['per_class_ap50']
        row.append(f'{ap[i]:.4f}' if i < len(ap) else 'N/A')
    ap_rows.append(row)
print(tabulate(ap_rows, headers=pc_headers, tablefmt='grid'))

# ─── Pilih Winner ────────────────────────────────────────────────────────────
WINNER = max(comparison, key=lambda vn: comparison[vn]['composite'])
WINNER_PT = progressive_results[WINNER]['best_pt']

print(f'\n★ WINNER: {WINNER.upper()} (composite={comparison[WINNER]["composite"]:.4f})')
print(f'  Path: {WINNER_PT}')

# ─── Bar Chart Visualization ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

variants = list(comparison.keys())
x = np.arange(len(CLASS_NAMES))
width = 0.35

for idx, metric_key in enumerate(['per_class_recall', 'per_class_prec', 'per_class_ap50']):
    ax = axes[idx]
    for j, vn in enumerate(variants):
        vals = comparison[vn][metric_key]
        ax.bar(x + j * width, vals[:len(CLASS_NAMES)], width, label=vn)
    ax.set_xticks(x + width)
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_title(metric_key.replace('per_class_', '').upper())
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Per-Class Metrics Comparison (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'progressive_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

with open(OUTPUT_DIR / 'comparison_results.json', 'w') as f:
    json.dump(comparison, f, indent=2, default=str)
print('\n✓ Comparison selesai')


Evaluasi yolov8n: /home/rof/Delima/Delima/Eksperimen 5/runs_v5/progressive_yolov8n/weights/best.pt
Ultralytics 8.4.22 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11908MiB)
Model summary (fused): 73 layers, 3,007,403 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 339.0±59.7 MB/s, size: 39.5 KB)
val: Scanning /home/rof/Delima/Delima/CLEAN DATASET /Balanced_Dataset/test/labels... 3367 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3367/3367 4.3Kit/s 0.8s0.1s
val: New cache created: /home/rof/Delima/Delima/CLEAN DATASET /Balanced_Dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 211/211 11.8it/s 17.8s0.2s
                   all       3367      15079      0.749      0.719      0.768       0.54
Speed: 0.5ms preprocess, 2.5ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /home/rof/Delima/Delima/CLEAN DATASET /runs/detec

<Figure size 1800x600 with 3 Axes>


✓ Comparison selesai


In [8]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from PIL import Image
from pathlib import Path
from ultralytics import YOLO
import wandb, math, random

# ─── Load winner model ────────────────────────────────────────────────────────
print(f'Evaluasi model terbaik: {WINNER.upper()}')
print(f'Path: {WINNER_PT}')

eval_model = YOLO(WINNER_PT)

# ─── Validasi pada TEST set ───────────────────────────────────────────────────
test_metrics = eval_model.val(
    data    = str(DATA_YAML),
    split   = 'test',
    batch   = 16,
    device  = DEVICE,
    plots   = True,
    save_json = True,
    verbose = True,
)

# ─── Summary ──────────────────────────────────────────────────────────────────
print('\n═══ FINAL TEST METRICS ═══')
print(f'  mAP50      : {test_metrics.box.map50:.4f}')
print(f'  mAP50-95   : {test_metrics.box.map:.4f}')
print(f'  Mean Recall: {np.mean(test_metrics.box.r):.4f}')
print(f'  Mean Prec  : {np.mean(test_metrics.box.p):.4f}')

print('\n─── Per-Class Recall ───')
for i, cls_name in enumerate(CLASS_NAMES):
    r = test_metrics.box.r[i] if i < len(test_metrics.box.r) else 0
    p = test_metrics.box.p[i] if i < len(test_metrics.box.p) else 0
    ap = test_metrics.box.ap50[i] if i < len(test_metrics.box.ap50) else 0
    print(f'  [{i}] {cls_name:25s}  R={r:.4f}  P={p:.4f}  AP50={ap:.4f}')

# ─── Validasi pada TRAIN set ──────────────────────────────────────────────────
print('\n─── Evaluasi pada TRAIN set ───')
train_metrics = eval_model.val(
    data    = str(DATA_YAML),
    split   = 'train',
    batch   = 16,
    device  = DEVICE,
    verbose = False,
)
print(f'  Train mAP50    : {train_metrics.box.map50:.4f}')
print(f'  Train mAP50-95 : {train_metrics.box.map:.4f}')

# ─── Inference Visual ─────────────────────────────────────────────────────────
test_img_dir = Data_Test / 'images'
test_lbl_dir = Data_Test / 'labels'

if test_img_dir.exists():
    all_images = sorted(list(test_img_dir.glob('*.jpg')) + list(test_img_dir.glob('*.png')))
    random.seed(42)
    sample_paths = random.sample(all_images, min(12, len(all_images)))

    COLORS = plt.cm.get_cmap('tab10', NUM_CLASSES)

    preds = eval_model.predict(
        source  = [str(p) for p in sample_paths],
        imgsz   = 640,
        conf    = 0.25,
        iou     = 0.45,
        device  = DEVICE,
        verbose = False,
    )

    n_cols = 4
    n_rows = math.ceil(len(sample_paths) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    axes = np.array(axes).flatten()

    wandb_images = []
    for idx, (img_path, result) in enumerate(zip(sample_paths, preds)):
        ax = axes[idx]
        img_arr = np.array(Image.open(img_path).convert('RGB'))
        ih, iw = img_arr.shape[:2]
        ax.imshow(img_arr)

        if test_lbl_dir and test_lbl_dir.exists():
            lbl_path = test_lbl_dir / (img_path.stem + '.txt')
            if lbl_path.exists():
                with open(lbl_path) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) < 5:
                            continue
                        cls_id = int(parts[0])
                        xc, yc, bw, bh = map(float, parts[1:5])
                        x1 = (xc - bw/2) * iw; y1 = (yc - bh/2) * ih
                        w = bw * iw; h = bh * ih
                        rect = patches.Rectangle((x1, y1), w, h,
                            linewidth=1.5, edgecolor='lime', facecolor='none',
                            linestyle='--')
                        ax.add_patch(rect)

        if result.boxes is not None and len(result.boxes) > 0:
            boxes_xyxy = result.boxes.xyxy.cpu().numpy()
            confs      = result.boxes.conf.cpu().numpy()
            cls_ids    = result.boxes.cls.cpu().numpy().astype(int)
            for (x1, y1, x2, y2), conf, cid in zip(boxes_xyxy, confs, cls_ids):
                color = COLORS(cid)
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                    linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1 - 4,
                        f'{CLASS_NAMES[cid]} {conf:.2f}',
                        fontsize=7, color='white',
                        bbox=dict(facecolor=color, alpha=0.7, pad=1, edgecolor='none'))

        ax.set_title(img_path.stem[:30], fontsize=8)
        ax.axis('off')
        wandb_images.append(wandb.Image(img_arr,
            caption=f'{img_path.name} | pred: {len(result.boxes) if result.boxes else 0}'))

    legend_lines = [
        Line2D([0], [0], color='lime', linestyle='--', lw=2, label='Ground Truth'),
        Line2D([0], [0], color='white', lw=2, label='Prediction'),
    ]
    fig.legend(handles=legend_lines, loc='lower center', ncol=2, fontsize=10)

    for j in range(len(sample_paths), len(axes)):
        axes[j].axis('off')

    plt.suptitle(f'Inference Visual: {WINNER.upper()} — GT (hijau putus) vs Pred (warna per kelas)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.savefig(str(OUTPUT_DIR / 'final_inference_visual.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Test image directory tidak ditemukan, skip inference visual.')
    wandb_images = []

# ─── WandB Upload ─────────────────────────────────────────────────────────────
eval_run = wandb.init(
    project  = WANDB_PROJECT,
    entity   = WANDB_ENTITY,
    name     = f'final_eval_{WINNER}',
    group    = 'evaluation',
    config   = {
        'winner': WINNER,
        'test_map50': test_metrics.box.map50,
        'test_map50_95': test_metrics.box.map,
    },
    reinit   = True,
)

eval_run.log({
    'test/mAP50':    test_metrics.box.map50,
    'test/mAP50-95': test_metrics.box.map,
    'test/recall':   float(np.mean(test_metrics.box.r)),
    'test/precision': float(np.mean(test_metrics.box.p)),
    'train/mAP50':    train_metrics.box.map50,
    'train/mAP50-95': train_metrics.box.map,
})

per_class_data = []
for i, cls_name in enumerate(CLASS_NAMES):
    eval_run.log({
        f'per_class/{cls_name}/recall':    float(test_metrics.box.r[i]),
        f'per_class/{cls_name}/precision': float(test_metrics.box.p[i]),
        f'per_class/{cls_name}/AP50':      float(test_metrics.box.ap50[i]),
    })

if wandb_images:
    eval_run.log({'inference_samples': wandb_images})

comp_png = OUTPUT_DIR / 'progressive_comparison.png'
if comp_png.exists():
    eval_run.log({'comparison_chart': wandb.Image(str(comp_png))})

inf_png = OUTPUT_DIR / 'final_inference_visual.png'
if inf_png.exists():
    eval_run.log({'inference_grid': wandb.Image(str(inf_png))})

eval_run.finish()
print('\n✓ Final evaluation selesai & diunggah ke WandB')

Evaluasi model terbaik: YOLOV8N
Path: /home/rof/Delima/Delima/Eksperimen 5/runs_v5/progressive_yolov8n/weights/best.pt
Ultralytics 8.4.22 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11908MiB)
Model summary (fused): 73 layers, 3,007,403 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2411.2±532.0 MB/s, size: 36.2 KB)
val: Scanning /home/rof/Delima/Delima/CLEAN DATASET /Balanced_Dataset/test/labels.cache... 3367 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3367/3367 1.2Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 211/211 10.4it/s 20.2s0.2s
                   all       3367      15079      0.749      0.719      0.768       0.54
         atrophic_scar        869       2797      0.664      0.618      0.654      0.273
                comedo       1302       2415       0.64      0.514      0.552      0.202
     hypertrophic_scar        163        740 

/tmp/ipykernel_1225571/2539691748.py:62: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  COLORS = plt.cm.get_cmap('tab10', NUM_CLASSES)


<Figure size 2000x1500 with 12 Axes>

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


per_class/atrophic_scar/AP50,▁
per_class/atrophic_scar/precision,▁
per_class/atrophic_scar/recall,▁
per_class/comedo/AP50,▁
per_class/comedo/precision,▁
per_class/comedo/recall,▁
per_class/hypertrophic_scar/AP50,▁
per_class/hypertrophic_scar/precision,▁
per_class/hypertrophic_scar/recall,▁
per_class/melasma/AP50,▁
+23,...



✓ Final evaluation selesai & diunggah ke WandB



✓ Final evaluation selesai & diunggah ke WandB


## Cell 8 — Final Evaluation TANPA Kelas `other`

Menghitung ulang semua metrik dengan mengecualikan kelas `other`
untuk melihat performa murni pada kelas-kelas lesi kulit yang relevan.

In [9]:
import numpy as np
from ultralytics import YOLO
from tabulate import tabulate
import matplotlib.pyplot as plt

# ─── Identifikasi indeks kelas 'other' ────────────────────────────────────────
OTHER_IDX = None
for i, name in enumerate(CLASS_NAMES):
    if name.lower() == 'other':
        OTHER_IDX = i
        break

if OTHER_IDX is None:
    print('Kelas "other" tidak ditemukan di CLASS_NAMES. Skip cell ini.')
else:
    print(f'Kelas "other" ditemukan di index {OTHER_IDX}')
    print(f'Mengevaluasi model terbaik: {WINNER.upper()} TANPA kelas other\n')

    eval_model_no_other = YOLO(WINNER_PT)

    test_metrics_full = eval_model_no_other.val(
        data    = str(DATA_YAML),
        split   = 'test',
        batch   = 16,
        device  = DEVICE,
        verbose = False,
    )

    keep_idx = [i for i in range(NUM_CLASSES) if i != OTHER_IDX]
    keep_names = [CLASS_NAMES[i] for i in keep_idx]

    recall_all  = test_metrics_full.box.r
    prec_all    = test_metrics_full.box.p
    ap50_all    = test_metrics_full.box.ap50
    ap50_95_all = test_metrics_full.box.ap

    recall_filtered  = [recall_all[i]  for i in keep_idx]
    prec_filtered    = [prec_all[i]    for i in keep_idx]
    ap50_filtered    = [ap50_all[i]    for i in keep_idx]
    ap50_95_filtered = [ap50_95_all[i] for i in keep_idx]

    mean_recall_no_other  = float(np.mean(recall_filtered))
    mean_prec_no_other    = float(np.mean(prec_filtered))
    map50_no_other        = float(np.mean(ap50_filtered))
    map50_95_no_other     = float(np.mean(ap50_95_filtered))
    min_recall_no_other   = float(np.min(recall_filtered))

    # ─── Overall Comparison: WITH vs WITHOUT 'other' ──────────────────────────
    print('='*70)
    print(' COMPARISON: ALL CLASSES vs WITHOUT "other"')
    print('='*70)

    comp_headers = ['Metric', 'All Classes', 'Without other', 'Delta']
    comp_rows = [
        ['mAP50',       f'{test_metrics_full.box.map50:.4f}', f'{map50_no_other:.4f}',
         f'{map50_no_other - test_metrics_full.box.map50:+.4f}'],
        ['mAP50-95',    f'{test_metrics_full.box.map:.4f}',   f'{map50_95_no_other:.4f}',
         f'{map50_95_no_other - test_metrics_full.box.map:+.4f}'],
        ['Mean Recall',  f'{float(np.mean(recall_all)):.4f}',  f'{mean_recall_no_other:.4f}',
         f'{mean_recall_no_other - float(np.mean(recall_all)):+.4f}'],
        ['Min Recall',   f'{float(np.min(recall_all)):.4f}',   f'{min_recall_no_other:.4f}',
         f'{min_recall_no_other - float(np.min(recall_all)):+.4f}'],
        ['Mean Precision', f'{float(np.mean(prec_all)):.4f}',  f'{mean_prec_no_other:.4f}',
         f'{mean_prec_no_other - float(np.mean(prec_all)):+.4f}'],
    ]
    print(tabulate(comp_rows, headers=comp_headers, tablefmt='grid'))

    # ─── Per-Class Detail (tanpa other) ───────────────────────────────────────
    print('\n' + '='*70)
    print(' PER-CLASS METRICS (WITHOUT "other")')
    print('='*70)

    pc_headers = ['Class', 'Recall', 'Precision', 'AP50', 'AP50-95']
    pc_rows = []
    for idx, ki in enumerate(keep_idx):
        pc_rows.append([
            CLASS_NAMES[ki],
            f'{recall_filtered[idx]:.4f}',
            f'{prec_filtered[idx]:.4f}',
            f'{ap50_filtered[idx]:.4f}',
            f'{ap50_95_filtered[idx]:.4f}',
        ])
    pc_rows.append(['─'*15, '─'*8, '─'*8, '─'*8, '─'*8])
    pc_rows.append([
        'MEAN',
        f'{mean_recall_no_other:.4f}',
        f'{mean_prec_no_other:.4f}',
        f'{map50_no_other:.4f}',
        f'{map50_95_no_other:.4f}',
    ])
    print(tabulate(pc_rows, headers=pc_headers, tablefmt='grid'))

    # ─── Visualisasi ──────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    x = np.arange(len(keep_names))

    for idx, (metric_vals, title) in enumerate([
        (recall_filtered, 'Recall'),
        (prec_filtered, 'Precision'),
        (ap50_filtered, 'AP50'),
    ]):
        ax = axes[idx]
        bars = ax.bar(x, metric_vals, color=plt.cm.Set2(np.linspace(0, 1, len(keep_names))))
        ax.axhline(y=np.mean(metric_vals), color='red', linestyle='--', alpha=0.7, label=f'Mean={np.mean(metric_vals):.3f}')
        ax.set_xticks(x)
        ax.set_xticklabels(keep_names, rotation=45, ha='right', fontsize=8)
        ax.set_ylim(0, 1.05)
        ax.set_title(title)
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        for bar, val in zip(bars, metric_vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7)

    plt.suptitle(f'{WINNER.upper()} — Per-Class Metrics WITHOUT "other" (Test Set)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'eval_without_other.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ─── WandB Upload ─────────────────────────────────────────────────────────
    no_other_run = wandb.init(
        project = WANDB_PROJECT,
        entity  = WANDB_ENTITY,
        name    = f'eval_no_other_{WINNER}',
        group   = 'evaluation',
        reinit  = True,
    )
    no_other_run.log({
        'no_other/mAP50':       map50_no_other,
        'no_other/mAP50-95':    map50_95_no_other,
        'no_other/mean_recall': mean_recall_no_other,
        'no_other/min_recall':  min_recall_no_other,
        'no_other/mean_prec':   mean_prec_no_other,
    })
    for idx, ki in enumerate(keep_idx):
        no_other_run.log({
            f'no_other/{CLASS_NAMES[ki]}/recall': recall_filtered[idx],
            f'no_other/{CLASS_NAMES[ki]}/precision': prec_filtered[idx],
            f'no_other/{CLASS_NAMES[ki]}/AP50': ap50_filtered[idx],
        })

    eval_png = OUTPUT_DIR / 'eval_without_other.png'
    if eval_png.exists():
        no_other_run.log({'eval_no_other_chart': wandb.Image(str(eval_png))})

    no_other_run.finish()
    print('\n✓ Evaluasi tanpa kelas "other" selesai & diunggah ke WandB')

Kelas "other" ditemukan di index 6
Mengevaluasi model terbaik: YOLOV8N TANPA kelas other

Ultralytics 8.4.22 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11908MiB)
Model summary (fused): 73 layers, 3,007,403 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1575.3±1179.6 MB/s, size: 46.4 KB)
val: Scanning /home/rof/Delima/Delima/CLEAN DATASET /Balanced_Dataset/test/labels.cache... 3367 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3367/3367 1.1Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 211/211 11.6it/s 18.2s0.2s
                   all       3367      15079      0.749      0.719      0.768       0.54
Speed: 0.5ms preprocess, 2.4ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /home/rof/Delima/Delima/CLEAN DATASET /runs/detect/val13
 COMPARISON: ALL CLASSES vs WITHOUT "other"
+----------------+---------------+-----------------

<Figure size 1800x600 with 3 Axes>

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


no_other/atrophic_scar/AP50,▁
no_other/atrophic_scar/precision,▁
no_other/atrophic_scar/recall,▁
no_other/comedo/AP50,▁
no_other/comedo/precision,▁
no_other/comedo/recall,▁
no_other/hypertrophic_scar/AP50,▁
no_other/hypertrophic_scar/precision,▁
no_other/hypertrophic_scar/recall,▁
no_other/mAP50,▁
+19,...



✓ Evaluasi tanpa kelas "other" selesai & diunggah ke WandB
